# AI Readiness Score — Fabric Notebook

This notebook connects to a **live Power BI semantic model** via `semantic-link` (sempy),
computes an **AI Readiness Score** by evaluating 7 categories — each scored independently
out of 100 — then takes a **simple average** to produce the overall 0–100 score.

**7 Categories (each scored 0–100):**
| # | Category | What We Check |
|---|---|---|
| 1 | Description Coverage | % of tables + columns + measures with non-empty descriptions |
| 2 | Naming Quality | % of objects with business-friendly names (no vw_, tbl_, GUIDs, ALL_CAPS) |
| 3 | Relationship Health | Penalty for BiDi (-15), M:M (-20), M:M+BiDi (-30) relationships |
| 4 | DAX Quality | Anti-patterns + Complexity + Format Strings + Calc-Column Ratio (avg of 4 sub-scores) |
| 5 | Column Metadata | 60% data-type coverage + 40% ID/key columns hidden |
| 6 | Model Structure | % of tables classifiable as Fact/Dim/Bridge/Parameter/Utility/etc. |
| 7 | Relationship Coverage | Ratio of relationships to tables + inactive relationship check |

**Grading:** A (90+), B (80+), C (70+), D (60+), F (<60)

**How to use:**
1. Upload this notebook to a Fabric workspace.
2. Set the parameters in Cell 2 (workspace name, dataset name, thresholds).
3. Run all cells.
4. Schedule via Fabric Pipeline for recurring alerts.

> Requires: `semantic-link` (pre-installed in Fabric), `sempy.fabric`


In [33]:
# ============================================================
# PARAMETERS — Set these before running
# ============================================================
WORKSPACE_NAME = "Your Fabric Workspace"      # Fabric workspace containing the model
DATASET_NAME   = "Your Semantic Model"             # Semantic model (dataset) name
ALERT_THRESHOLD = 60                         # Score below this triggers an alert
LOG_TO_LAKEHOUSE = False                     # Set True to log scores to a Lakehouse table
# LAKEHOUSE_NAME = "Your Lakehouse"            # Lakehouse name (if logging enabled)
SEND_EMAIL_ALERT = True                      # Set True to send email via Fabric


StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 35, Finished, Available, Finished, False)

In [34]:
# ============================================================
# Install / import dependencies
# ============================================================
import sempy.fabric as fabric
import pandas as pd
import re
import json
from datetime import datetime
from IPython.display import display, HTML, Markdown

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 36, Finished, Available, Finished, False)

In [35]:
# ============================================================
# Fetch model metadata via sempy
# ============================================================
print(f"Connecting to: {WORKSPACE_NAME} / {DATASET_NAME}")

# Tables and their properties
df_tables = fabric.list_tables(DATASET_NAME, workspace=WORKSPACE_NAME)
print(f"Tables: {len(df_tables)}")

# Columns with data types, descriptions, hidden flags
df_columns = fabric.list_columns(DATASET_NAME, workspace=WORKSPACE_NAME)
print(f"Columns: {len(df_columns)}")

# Measures
df_measures = fabric.list_measures(DATASET_NAME, workspace=WORKSPACE_NAME)
print(f"Measures: {len(df_measures)}")

# Relationships
df_relationships = fabric.list_relationships(DATASET_NAME, workspace=WORKSPACE_NAME)
print(f"Relationships: {len(df_relationships)}")

# Hierarchies (if available)
try:
    df_hierarchies = fabric.list_hierarchies(DATASET_NAME, workspace=WORKSPACE_NAME)
    print(f"Hierarchies: {len(df_hierarchies)}")
except Exception:
    df_hierarchies = pd.DataFrame()
    print("Hierarchies: N/A")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 37, Finished, Available, Finished, False)

Connecting to: Your Fabric Workspace / Your Semantic Model
Tables: 51
Columns: 328
Measures: 32
Relationships: 103
Hierarchies: 48


In [36]:
# ============================================================
# Scoring functions
# ============================================================
# Each category is scored independently out of 100.
# Overall score = simple average of all 7 categories.
CATEGORIES_LIST = [
    "Description Coverage",
    "Naming Quality",
    "Relationship Health",
    "DAX Quality",
    "Column Metadata",
    "Model Structure",
    "Relationship Coverage",
]
NUM_CATEGORIES = len(CATEGORIES_LIST)

def is_business_friendly(name):
    """Check if a name follows business-friendly conventions."""
    if not name or len(name) <= 2:
        return False
    if re.match(r'^(vw|tbl|dim_|fact_|stg_|src_|dbo_|raw_)', name, re.IGNORECASE):
        return False
    if name == name.upper() and len(name) > 3:
        return False
    if name.count('_') > 2:
        return False
    # Reject GUID-like names (auto-generated tables)
    if re.search(r'[0-9a-f]{8}-[0-9a-f]{4}', name, re.IGNORECASE):
        return False
    return True

def classify_table(name):
    """Classify a table into a semantic role."""
    n = name.lower()
    if n.startswith('localdatetable_') or n.startswith('datetabletemplate_'):
        return 'Auto-generated'
    if 'bridge' in n:
        return 'Bridge'
    if n.startswith('parameter'):
        return 'Field Parameter'
    if 'measures' in n and 'table' in n:
        return 'Measures Only'
    if n == 'datarefresh' or 'refresh' in n or 'freshness' in n:
        return 'Utility'
    if 'date' in n or 'calendar' in n:
        return 'Date Dimension'
    if 'fact' in n:
        return 'Fact'
    if 'dim' in n:
        return 'Dimension'
    return 'Other'

def has_description(desc):
    """Check if a description value is non-empty."""
    if desc is None:
        return False
    if isinstance(desc, float):
        import math
        if math.isnan(desc):
            return False
    return str(desc).strip() != ''

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 38, Finished, Available, Finished, False)

In [37]:
# ============================================================
# Score 1: Description Coverage
# ============================================================
desc_col = 'Description' if 'Description' in df_tables.columns else 'description'
name_col_tables = 'Name' if 'Name' in df_tables.columns else 'name'

# Tables
tables_with_desc = df_tables[desc_col].apply(has_description).sum() if desc_col in df_tables.columns else 0
tables_total = len(df_tables)
tables_missing = []
if desc_col in df_tables.columns:
    for _, row in df_tables.iterrows():
        if not has_description(row.get(desc_col)):
            tables_missing.append(row.get(name_col_tables, '?'))
else:
    tables_missing = list(df_tables[name_col_tables])

# Columns — auto-detect column names
col_desc_col = None
col_name_col = None
col_table_col = None
for c in df_columns.columns:
    cl = c.lower().replace(' ', '').replace('_', '')
    if cl in ('description',) and col_desc_col is None: col_desc_col = c
    elif cl in ('columnname', 'name') and col_name_col is None: col_name_col = c
    elif cl in ('tablename', 'table') and col_table_col is None: col_table_col = c
if col_desc_col is None:
    for c in df_columns.columns:
        if 'desc' in c.lower(): col_desc_col = c; break

cols_with_desc = df_columns[col_desc_col].apply(has_description).sum() if col_desc_col and col_desc_col in df_columns.columns else 0
cols_total = len(df_columns)
cols_missing = []
if col_desc_col and col_desc_col in df_columns.columns and col_name_col:
    for _, row in df_columns.iterrows():
        if not has_description(row.get(col_desc_col)):
            tbl = row.get(col_table_col, '') if col_table_col else ''
            cols_missing.append(f"{tbl}.{row.get(col_name_col, '?')}")
elif col_name_col:
    for _, row in df_columns.iterrows():
        tbl = row.get(col_table_col, '') if col_table_col else ''
        cols_missing.append(f"{tbl}.{row.get(col_name_col, '?')}")

# Measures — auto-detect column names
meas_desc_col = None
meas_name_col = None
meas_table_col = None
for c in df_measures.columns:
    cl = c.lower().replace(' ', '').replace('_', '')
    if cl in ('description',) and meas_desc_col is None: meas_desc_col = c
    elif cl in ('measurename', 'name') and meas_name_col is None: meas_name_col = c
    elif cl in ('tablename', 'table') and meas_table_col is None: meas_table_col = c
if meas_desc_col is None:
    for c in df_measures.columns:
        if 'desc' in c.lower(): meas_desc_col = c; break
if meas_name_col is None and len(df_measures.columns) > 1:
    meas_name_col = df_measures.columns[1]

meas_with_desc = df_measures[meas_desc_col].apply(has_description).sum() if meas_desc_col and meas_desc_col in df_measures.columns else 0
meas_total = len(df_measures)
meas_missing = []
if meas_desc_col and meas_desc_col in df_measures.columns and meas_name_col:
    for _, row in df_measures.iterrows():
        if not has_description(row.get(meas_desc_col)):
            tbl = row.get(meas_table_col, '') if meas_table_col else ''
            meas_missing.append(f"{tbl}.{row.get(meas_name_col, '?')}")
elif meas_name_col:
    # Description column not found — all measures are missing descriptions
    for _, row in df_measures.iterrows():
        tbl = row.get(meas_table_col, '') if meas_table_col else ''
        meas_missing.append(f"{tbl}.{row.get(meas_name_col, '?')}")

total_objects = tables_total + cols_total + meas_total
described_objects = int(tables_with_desc + cols_with_desc + meas_with_desc)
desc_score = (described_objects / max(total_objects, 1)) * 100
desc_missing_count = total_objects - described_objects

desc_suggestions = []
if desc_missing_count > 0:
    desc_suggestions.append(f"{desc_missing_count} of {total_objects} objects are missing descriptions.")

print(f"Description Coverage: {described_objects}/{total_objects} = {desc_score:.0f}/100")
print(f"  Tables: {int(tables_with_desc)}/{tables_total} | Columns: {int(cols_with_desc)}/{cols_total} | Measures: {int(meas_with_desc)}/{meas_total}")

# Show ALL missing objects (like Column Metadata shows visible ID columns)
all_missing_objects = []
for t in tables_missing:
    all_missing_objects.append(f"[Table] {t}")
for c in cols_missing:
    all_missing_objects.append(f"[Column] {c}")
for m in meas_missing:
    all_missing_objects.append(f"[Measure] {m}")

if all_missing_objects:
    print(f"\n  Objects missing descriptions ({len(all_missing_objects)}):")
    for obj in all_missing_objects:
        print(f"    - {obj}")
else:
    print("\n  All objects have descriptions.")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 39, Finished, Available, Finished, False)

Description Coverage: 411/411 = 100/100
  Tables: 51/51 | Columns: 328/328 | Measures: 32/32

  All objects have descriptions.


In [38]:
# ============================================================
# Score 2: Naming Quality
# ============================================================
name_col_tables = 'Name' if 'Name' in df_tables.columns else 'name'
name_col_cols = 'Column Name' if 'Column Name' in df_columns.columns else 'Name' if 'Name' in df_columns.columns else 'name'
name_col_meas = 'Measure Name' if 'Measure Name' in df_measures.columns else 'Name' if 'Name' in df_measures.columns else 'name'
col_table_col2 = 'Table Name' if 'Table Name' in df_columns.columns else 'table'
meas_table_col2 = 'Table Name' if 'Table Name' in df_measures.columns else 'table'

all_names = (
    list(df_tables[name_col_tables]) +
    list(df_columns[name_col_cols]) +
    list(df_measures[name_col_meas])
)
good_names = sum(1 for n in all_names if is_business_friendly(str(n)))
naming_score = (good_names / max(len(all_names), 1)) * 100
bad_count = len(all_names) - good_names

# Collect all objects with unfriendly names
bad_name_objects = []
for _, row in df_tables.iterrows():
    n = str(row.get(name_col_tables, ''))
    if not is_business_friendly(n):
        bad_name_objects.append(f"[Table] {n}")

for _, row in df_columns.iterrows():
    n = str(row.get(name_col_cols, ''))
    if not is_business_friendly(n):
        tbl = row.get(col_table_col2, '') if col_table_col2 in df_columns.columns else ''
        bad_name_objects.append(f"[Column] {tbl}.{n}")

for _, row in df_measures.iterrows():
    n = str(row.get(name_col_meas, ''))
    if not is_business_friendly(n):
        tbl = row.get(meas_table_col2, '') if meas_table_col2 in df_measures.columns else ''
        bad_name_objects.append(f"[Measure] {tbl}.{n}")

naming_suggestions = []
if bad_count > 0:
    naming_suggestions.append(f"{bad_count} objects have technical or unfriendly names.")

print(f"Naming Quality: {good_names}/{len(all_names)} = {naming_score:.0f}/100")

if bad_name_objects:
    print(f"\n  Objects with unfriendly names ({len(bad_name_objects)}):")
    for obj in bad_name_objects:
        print(f"    - {obj}")
else:
    print("\n  All objects have business-friendly names.")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 40, Finished, Available, Finished, False)

Naming Quality: 378/411 = 92/100

  Objects with unfriendly names (33):
    - [Table] vwfactcommercialactuals_current
    - [Table] vwdimplanningcategory
    - [Table] vwDimIsPartner
    - [Table] vwDimIsStrategicCohort
    - [Table] vwDimDataSource
    - [Table] vwDimCustomerLanguage
    - [Table] vwDimServiceOffering
    - [Table] vwDimCustomerArea
    - [Table] vwDimPESProduct
    - [Table] vwDimProductResponsibility
    - [Table] vwDimCloudProductsFilter
    - [Table] vwfactcommercialactuals_p
    - [Table] vwfactcommercialactuals_p2p
    - [Table] vwfactcommercialactuals_p2p2p
    - [Table] vwDimDate
    - [Table] vwfactcommercialactuals_current_agg
    - [Table] vwfactcommercialactuals_p_agg
    - [Table] vwfactcommercialactuals_p2p_agg
    - [Table] vwfactcommercialactuals_p2p2p_agg
    - [Table] vwdimplanningcategory_bridge
    - [Table] vwSAPtoSubPlanningCategory
    - [Table] vwDimSupportArea
    - [Table] vwMWPSubPCYRules
    - [Table] vwASMSSubPCYRules
    - [Table] vwfactm

In [39]:
# ============================================================
# Score 3a: Relationship Coverage 
# ============================================================
num_rels = len(df_relationships)
num_tables = len(df_tables)

rel_ratio = num_rels / max(num_tables - 1, 1)
rel_coverage_score = min(100, rel_ratio * 100)

rel_cov_suggestions = []
if num_rels == 0:
    rel_cov_suggestions.append("No relationships defined. Define star-schema relationships.")
elif rel_ratio < 0.5:
    rel_cov_suggestions.append(f"Only {num_rels} relationships for {num_tables} tables. Many tables may be disconnected.")

# Check for inactive relationships
inactive_count = 0
if 'Is Active' in df_relationships.columns:
    inactive_count = int((~df_relationships['Is Active']).sum())
    if inactive_count > 0:
        rel_cov_suggestions.append(f"{inactive_count} inactive relationship(s) — review if needed.")

print(f"Relationship Coverage: {num_rels} rels / {num_tables} tables = {rel_coverage_score:.0f}%")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 41, Finished, Available, Finished, False)

Relationship Coverage: 103 rels / 51 tables = 100%


In [40]:
# ============================================================
# Score 3b: Relationship Health — BiDi / M:M penalty
# ============================================================
bidi_count = 0
mm_count = 0
mm_bidi_count = 0
rel_health_suggestions = []

# Debug: print actual columns so we know what sempy returns
print(f"Relationship columns available: {list(df_relationships.columns)}")

# Auto-detect column names (sempy uses different names across versions)
cross_col = None
from_card_col = None
to_card_col = None

for col in df_relationships.columns:
    cl = col.lower().replace(' ', '').replace('_', '')
    if 'crossfilter' in cl or 'crossfilteringbehavior' in cl or 'filterdirection' in cl or cl == 'crossfilteringbehavior':
        cross_col = col
    elif 'fromcardinality' in cl or 'frommultiplicity' in cl:
        from_card_col = col
    elif 'tocardinality' in cl or 'tomultiplicity' in cl:
        to_card_col = col
    elif cl == 'multiplicity' and from_card_col is None:
        from_card_col = col  # fallback

print(f"Detected: cross_col={cross_col}, from_card={from_card_col}, to_card={to_card_col}")

if cross_col and from_card_col and to_card_col:
    for _, r in df_relationships.iterrows():
        cross_val = str(r.get(cross_col, '')).lower()
        is_bidi = 'both' in cross_val or 'bidirectional' in cross_val
        fc = str(r.get(from_card_col, '')).lower()
        tc = str(r.get(to_card_col, '')).lower()
        is_mm = 'many' in fc and 'many' in tc

        if is_mm and is_bidi:
            mm_bidi_count += 1
        elif is_mm:
            mm_count += 1
        elif is_bidi:
            bidi_count += 1

    total_bad = bidi_count + mm_count + mm_bidi_count
    if total_bad == 0:
        rel_health_score = 100.0
    else:
        penalty = mm_bidi_count * 30 + mm_count * 20 + bidi_count * 15
        rel_health_score = max(0, 100 - penalty)

    if mm_bidi_count > 0:
        rel_health_suggestions.append(f"CRITICAL: {mm_bidi_count} M:M + BiDi relationship(s).")
    if mm_count > 0:
        rel_health_suggestions.append(f"{mm_count} Many-to-Many relationship(s) — causes duplicate rows.")
    if bidi_count > 0:
        rel_health_suggestions.append(f"{bidi_count} BiDi relationship(s) — ambiguous filter paths.")
    if total_bad == 0:
        rel_health_suggestions.append("No BiDi or M:M relationships — excellent.")

elif len(df_relationships) > 0:
    # Fallback: try reading first row to understand structure
    print(f"First relationship row: {df_relationships.iloc[0].to_dict()}")
    # Use a heuristic: check ALL columns for 'Both'/'Many' values
    found_bidi = False
    found_mm = False
    for col in df_relationships.columns:
        vals = df_relationships[col].astype(str).str.lower().unique()
        if any('both' in v for v in vals):
            cross_col = col
            found_bidi = True
        if any('many' in v for v in vals):
            if from_card_col is None:
                from_card_col = col
            elif to_card_col is None:
                to_card_col = col

    if found_bidi and cross_col:
        bidi_count = int(df_relationships[cross_col].astype(str).str.lower().str.contains('both').sum())

    if from_card_col and to_card_col:
        for _, r in df_relationships.iterrows():
            fc = str(r.get(from_card_col, '')).lower()
            tc = str(r.get(to_card_col, '')).lower()
            if 'many' in fc and 'many' in tc:
                mm_count += 1

    total_bad = bidi_count + mm_count
    if total_bad == 0:
        rel_health_score = 100.0
    else:
        penalty = mm_count * 20 + bidi_count * 15
        rel_health_score = max(0, 100 - penalty)

    if mm_count > 0:
        rel_health_suggestions.append(f"{mm_count} M:M relationship(s) detected via heuristic.")
    if bidi_count > 0:
        rel_health_suggestions.append(f"{bidi_count} BiDi relationship(s) detected via heuristic.")
    if total_bad == 0:
        rel_health_suggestions.append("No BiDi or M:M detected.")
else:
    rel_health_score = 100.0
    rel_health_suggestions.append("No relationships to check.")

print(f"Relationship Health: BiDi={bidi_count}, M:M={mm_count}, M:M+BiDi={mm_bidi_count} -> {rel_health_score:.0f}/100")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 42, Finished, Available, Finished, False)

Relationship columns available: ['Multiplicity', 'From Table', 'From Column', 'To Table', 'To Column', 'Active', 'Cross Filtering Behavior', 'Security Filtering Behavior', 'Join On Date Behavior', 'Rely On Referential Integrity', 'State', 'Modified Time', 'Relationship Name']
Detected: cross_col=Cross Filtering Behavior, from_card=Multiplicity, to_card=None
First relationship row: {'Multiplicity': 'm:1', 'From Table': 'vwfactcommercialactuals_current', 'From Column': 'SubPlanningCategoryID', 'To Table': 'vwdimplanningcategory_bridge', 'To Column': 'PlanningCategoryID', 'Active': True, 'Cross Filtering Behavior': 'OneDirection', 'Security Filtering Behavior': 'OneDirection', 'Join On Date Behavior': 'DateAndTime', 'Rely On Referential Integrity': False, 'State': 'Ready', 'Modified Time': Timestamp('2026-04-09 17:00:51'), 'Relationship Name': '10fbcccd-3c8e-47c9-9a3b-902f2784753d'}
Relationship Health: BiDi=1, M:M=0, M:M+BiDi=0 -> 85/100


In [41]:
# ============================================================
# Score 4: DAX Quality — anti-patterns, complexity, format, calc-col ratio
# ============================================================
# Covers measures AND calculated columns. No overlap with other scores.
# 4 sub-checks, each scored 0–100, then simple average (total / 4).

print(f"Measure columns available: {list(df_measures.columns)}")
print(f"Column columns available: {list(df_columns.columns)}")

# --- Auto-detect measure columns ---
name_col_meas = None
expr_col = None
fmt_col = None
for col in df_measures.columns:
    cl = col.lower().replace(' ', '').replace('_', '')
    if cl in ('measurename', 'name') and name_col_meas is None:
        name_col_meas = col
    if cl in ('measureexpression', 'expression') and expr_col is None:
        expr_col = col
    if cl in ('formatstring',) and fmt_col is None:
        fmt_col = col
if name_col_meas is None:
    name_col_meas = df_measures.columns[1] if len(df_measures.columns) > 1 else df_measures.columns[0]

# --- Auto-detect calculated columns ---
col_type_col = None
col_expr_col = None
col_name_col2 = None
col_table_col2 = None
for col in df_columns.columns:
    cl = col.lower().replace(' ', '').replace('_', '')
    if cl in ('columntype', 'type') and col_type_col is None:
        col_type_col = col
    if cl in ('expression', 'columnexpression') and col_expr_col is None:
        col_expr_col = col
    if cl in ('columnname', 'name') and col_name_col2 is None:
        col_name_col2 = col
    if cl in ('tablename', 'table') and col_table_col2 is None:
        col_table_col2 = col

print(f"Detected measure: name={name_col_meas}, expr={expr_col}, fmt={fmt_col}")
print(f"Detected column: type={col_type_col}, expr={col_expr_col}, name={col_name_col2}")

# --- Collect all DAX expressions (measures + calculated columns) ---
dax_items = []

for _, row in df_measures.iterrows():
    mname = str(row.get(name_col_meas, ''))
    mexpr = str(row.get(expr_col, '')) if expr_col else ''
    dax_items.append((f"[Measure] {mname}", mexpr))

calc_col_count = 0
total_col_count = len(df_columns)
if col_type_col and col_expr_col:
    for _, row in df_columns.iterrows():
        ctype = str(row.get(col_type_col, '')).lower()
        if 'calculated' in ctype or 'calc' in ctype:
            calc_col_count += 1
            cname = str(row.get(col_name_col2, ''))
            tname = str(row.get(col_table_col2, ''))
            cexpr = str(row.get(col_expr_col, ''))
            dax_items.append((f"[CalcCol] {tname}.{cname}", cexpr))

print(f"\nTotal DAX items: {len(dax_items)} ({meas_total} measures + {calc_col_count} calc columns)")

# ================================================================
# Anti-pattern checks
# ================================================================
def detect_antipatterns(expression):
    if not expression or expression.strip() == '':
        return []
    dax = expression
    issues = []
    if re.findall(r'(?<![/"\'])\/(?![/\*])', dax):
        issues.append("uses / instead of DIVIDE()")
    if re.search(r'\bFILTER\s*\(\s*(?!ALL\b|VALUES\b|DISTINCT\b|ADDCOLUMNS\b|SELECTCOLUMNS\b|SUMMARIZE\b|UNION\b|DATATABLE\b|FILTER\b|GENERATESERIES\b)[A-Z][A-Za-z_\s]+,', dax, re.IGNORECASE):
        issues.append("FILTER on full table")
    if len(re.findall(r'\bIF\s*\(', dax, re.IGNORECASE)) > 3:
        issues.append("nested IF — use SWITCH")
    if re.search(r'\bEARLIER\b|\bEARLIEST\b', dax, re.IGNORECASE):
        issues.append("EARLIER/EARLIEST — use VAR")
    if re.search(r'\bCALCULATE\s*\([^)]*\bCALCULATE\s*\(', dax, re.IGNORECASE):
        issues.append("nested CALCULATE")
    if re.search(r'\bCOUNTROWS\s*\(\s*FILTER\s*\(', dax, re.IGNORECASE):
        issues.append("COUNTROWS(FILTER(...))")
    if re.search(r'\bCALCULATE\s*\(.*\bFILTER\s*\(\s*ALL\s*\(', dax, re.IGNORECASE | re.DOTALL):
        issues.append("FILTER(ALL(...)) — use REMOVEFILTERS")
    if re.search(r'\b(SUMX|AVERAGEX|MAXX|MINX|RANKX|PRODUCTX|CONCATENATEX)\s*\(\s*(?!ALL\b|VALUES\b|DISTINCT\b|FILTER\b|ADDCOLUMNS\b|SELECTCOLUMNS\b|SUMMARIZE\b|TOPN\b|GENERATESERIES\b)[A-Z][A-Za-z_\s]+,', dax, re.IGNORECASE):
        issues.append("iterator on unfiltered table")
    if len(dax) > 200 and not re.search(r'\bVAR\b', dax, re.IGNORECASE):
        issues.append("complex DAX without VAR/RETURN")
    return issues

# ================================================================
# Complexity checks
# ================================================================
def detect_complexity(expression):
    if not expression or expression.strip() == '':
        return []
    dax = expression
    issues = []
    max_depth = 0
    depth = 0
    for ch in dax:
        if ch == '(':
            depth += 1
            max_depth = max(max_depth, depth)
        elif ch == ')':
            depth -= 1
    if max_depth > 8:
        issues.append(f"deep nesting ({max_depth} levels)")
    elif max_depth > 5:
        issues.append(f"moderate nesting ({max_depth} levels)")
    if len(re.findall(r'\bCALCULATE\b', dax, re.IGNORECASE)) > 3:
        issues.append("too many CALCULATE calls")
    if len(re.findall(r'\bFILTER\b', dax, re.IGNORECASE)) > 3:
        issues.append("too many FILTER calls")
    if len(dax) > 1000:
        issues.append("very long (>1000 chars)")
    elif len(dax) > 500:
        issues.append("long (>500 chars)")
    return issues

# ================================================================
# Run checks — count clean vs flagged
# ================================================================
ap_clean = 0
cx_clean = 0
all_dax_issues = []

for label, expr in dax_items:
    ap_issues = detect_antipatterns(expr)
    cx_issues = detect_complexity(expr)
    if not ap_issues:
        ap_clean += 1
    if not cx_issues:
        cx_clean += 1
    if ap_issues or cx_issues:
        combined = ap_issues + cx_issues
        all_dax_issues.append(f"{label}: {', '.join(combined)}")

total_dax = len(dax_items) if dax_items else 1
ap_flagged = total_dax - ap_clean
cx_flagged = total_dax - cx_clean

# --- Sub-score 1: Anti-patterns (0–100) ---
antipattern_score = (ap_clean / total_dax) * 100

# --- Sub-score 2: Complexity (0–100) ---
complexity_score = (cx_clean / total_dax) * 100

# --- Sub-score 3: Format String (0–100) ---
fmt_good = 0
fmt_bad_list = []
for _, row in df_measures.iterrows():
    mname = str(row.get(name_col_meas, ''))
    fmt_val = row.get(fmt_col, None) if fmt_col else None
    if fmt_val is not None and str(fmt_val).strip() != '':
        fmt_good += 1
    else:
        fmt_bad_list.append(mname)
fmt_bad = meas_total - fmt_good
fmt_score = (fmt_good / max(meas_total, 1)) * 100

# --- Sub-score 4: Calc-Col Ratio (0–100) ---
if total_col_count > 0:
    calc_col_score = ((total_col_count - calc_col_count) / total_col_count) * 100
else:
    calc_col_score = 100

# --- Final: simple average of 4 sub-scores ---
dax_quality_score = (antipattern_score + complexity_score + fmt_score + calc_col_score) / 4

# Build suggestions
dax_suggestions = []
if fmt_bad > 0:
    dax_suggestions.append(f"{fmt_bad} of {meas_total} measures have no format string.")
    for m in fmt_bad_list[:3]:
        dax_suggestions.append(f"  - {m} (no format string)")
if calc_col_count > 0:
    dax_suggestions.append(f"{calc_col_count} calculated column(s) — consider moving to measures or Power Query.")
if all_dax_issues:
    dax_suggestions.append(f"{len(all_dax_issues)} DAX expression(s) have issues:")
    for issue in all_dax_issues[:5]:
        dax_suggestions.append(f"  - {issue}")

print(f"\nDAX Quality: {dax_quality_score:.0f}/100  (simple average of 4 sub-scores)")
print(f"  1. Anti-patterns:  {ap_clean}/{total_dax} clean, {ap_flagged} flagged → {antipattern_score:.0f}/100")
print(f"  2. Complexity:     {cx_clean}/{total_dax} clean, {cx_flagged} flagged → {complexity_score:.0f}/100")
print(f"  3. Format String:  {fmt_good}/{meas_total} have format → {fmt_score:.0f}/100")
print(f"  4. Calc-Col Ratio: {calc_col_count}/{total_col_count} are calc cols → {calc_col_score:.0f}/100")

if all_dax_issues:
    print(f"\n  DAX issues ({len(all_dax_issues)}):")
    for issue in all_dax_issues[:10]:
        print(f"    - {issue}")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 43, Finished, Available, Finished, False)

Measure columns available: ['Table Name', 'Measure Name', 'Measure Expression', 'Measure Data Type', 'Measure Hidden', 'Measure Display Folder', 'Measure Description', 'Format String', 'Data Category', 'Detail Rows Definition', 'Format String Definition']
Column columns available: ['Table Name', 'Column Name', 'Description', 'Type', 'Data Type', 'Hidden', 'Format String', 'Source', 'Data Category', 'Display Folder', 'Key', 'Unique', 'Sort By Column', 'Summarize By', 'Is Available in MDX', 'Encoding Hint', 'State', 'Error Message', 'Alternate Of Base Column', 'Alternate Of Base Table', 'Modified Time']
Detected measure: name=Measure Name, expr=Measure Expression, fmt=Format String
Detected column: type=Type, expr=None, name=Column Name

Total DAX items: 32 (32 measures + 0 calc columns)

DAX Quality: 81/100  (simple average of 4 sub-scores)
  1. Anti-patterns:  19/32 clean, 13 flagged → 59/100
  2. Complexity:     32/32 clean, 0 flagged → 100/100
  3. Format String:  21/32 have format →

In [42]:
# ============================================================
# Score 5: Column Metadata
# ============================================================
print(f"Column DataFrame columns: {list(df_columns.columns)}")

# Auto-detect column names
dtype_col = None
hidden_col = None
col_name_col = None
table_name_col = None

for col in df_columns.columns:
    cl = col.lower().replace(' ', '').replace('_', '')
    if cl in ('datatype', 'type') and dtype_col is None:
        dtype_col = col
    elif cl in ('ishidden', 'hidden') and hidden_col is None:
        hidden_col = col
    elif cl in ('columnname', 'name') and col_name_col is None:
        col_name_col = col
    elif cl in ('tablename', 'table') and table_name_col is None:
        table_name_col = col

print(f"Detected: dtype={dtype_col}, hidden={hidden_col}, col_name={col_name_col}, table_name={table_name_col}")

cols_with_type = 0
if dtype_col and dtype_col in df_columns.columns:
    cols_with_type = int(df_columns[dtype_col].notna().sum())

# Find ID/key columns and check if hidden
id_total = 0
hidden_id = 0
visible_id_list = []

if col_name_col and col_name_col in df_columns.columns:
    for _, row in df_columns.iterrows():
        cname = str(row.get(col_name_col, ''))
        if re.search(r'(id|key|sk|fk)$', cname, re.IGNORECASE):
            id_total += 1
            is_hidden = bool(row.get(hidden_col, False)) if hidden_col else False
            tname = str(row.get(table_name_col, '')) if table_name_col else ''
            if is_hidden:
                hidden_id += 1
            else:
                visible_id_list.append(f"{tname}.{cname}")

type_pct = (cols_with_type / max(cols_total, 1)) * 100
hidden_pct = (hidden_id / max(id_total, 1)) * 100 if id_total > 0 else 100
column_meta_score = type_pct * 0.6 + hidden_pct * 0.4

col_suggestions = []
if type_pct < 100:
    col_suggestions.append(f"{cols_total - cols_with_type} columns have unknown data types.")
if id_total > 0 and hidden_pct < 100:
    col_suggestions.append(f"{id_total - hidden_id} ID/key columns are visible — consider hiding them.")

print(f"\nColumn Metadata Score: {column_meta_score:.0f}/100")
print(f"  Data types: {cols_with_type}/{cols_total} = {type_pct:.0f}% (×0.6 = {type_pct*0.6:.0f})")
print(f"  ID/key hidden: {hidden_id}/{id_total} = {hidden_pct:.0f}% (×0.4 = {hidden_pct*0.4:.0f})")

if visible_id_list:
    print(f"\n  Visible ID/key columns that should be hidden ({len(visible_id_list)}):")
    for v in visible_id_list:
        print(f"    - {v}")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 44, Finished, Available, Finished, False)

Column DataFrame columns: ['Table Name', 'Column Name', 'Description', 'Type', 'Data Type', 'Hidden', 'Format String', 'Source', 'Data Category', 'Display Folder', 'Key', 'Unique', 'Sort By Column', 'Summarize By', 'Is Available in MDX', 'Encoding Hint', 'State', 'Error Message', 'Alternate Of Base Column', 'Alternate Of Base Table', 'Modified Time']
Detected: dtype=Type, hidden=Hidden, col_name=Column Name, table_name=Table Name

Column Metadata Score: 97/100
  Data types: 328/328 = 100% (×0.6 = 60)
  ID/key hidden: 42/45 = 93% (×0.4 = 37)

  Visible ID/key columns that should be hidden (3):
    - vwSAPtoSubPlanningCategory.SAPId
    - vwSAPtoSubPlanningCategory.PrevSubPCYId
    - vwSAPtoSubPlanningCategory.CurrSubPCYId


In [43]:
# ============================================================
# Score 6: Model Structure (10%)
# ============================================================
table_names = list(df_tables[name_col_tables])
classifications = {n: classify_table(str(n)) for n in table_names}

# Count by type
type_counts = {}
for v in classifications.values():
    type_counts[v] = type_counts.get(v, 0) + 1

facts = type_counts.get('Fact', 0)
dims = type_counts.get('Dimension', 0) + type_counts.get('Date Dimension', 0)
bridges = type_counts.get('Bridge', 0)
params = type_counts.get('Field Parameter', 0)
auto = type_counts.get('Auto-generated', 0)
utility = type_counts.get('Utility', 0) + type_counts.get('Measures Only', 0)
refs = type_counts.get('Reference/Mapping', 0)
other = type_counts.get('Other', 0)

# All non-Other tables are classifiable (auto-gen, params, utility all count)
classifiable = len(table_names) - other
classifiable_ratio = classifiable / max(len(table_names), 1)
structure_score = classifiable_ratio * 100

struct_suggestions = []
if other > 0:
    other_names = [n for n, v in classifications.items() if v == 'Other']
    struct_suggestions.append(f"{other} table(s) unclassified: {', '.join(other_names[:5])}")
if facts == 0:
    struct_suggestions.append("No fact tables detected.")
if dims == 0:
    struct_suggestions.append("No dimension tables detected.")
if auto > 0:
    struct_suggestions.append(f"{auto} auto-generated date table(s) — consider removing and using a proper date dimension.")

print(f"Model Structure: {facts}F / {dims}D / {bridges}B / {params}FP / {auto}Auto / {utility}Util / {refs}Ref / {other}Other = {structure_score:.0f}%")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 45, Finished, Available, Finished, False)

Model Structure: 12F / 14D / 2B / 12FP / 0Auto / 3Util / 3Ref / 5Other = 90%


In [44]:
# ============================================================
# Aggregate scores — Simple average (each category out of 100)
# ============================================================
categories = [
    ("Description Coverage",    desc_score,           desc_suggestions),
    ("Naming Quality",          naming_score,          naming_suggestions),
    ("Relationship Health",     rel_health_score,      rel_health_suggestions),
    ("DAX Quality",         dax_quality_score,         dax_suggestions),
    ("Column Metadata",         column_meta_score,     col_suggestions),
    ("Model Structure",         structure_score,       struct_suggestions),
    ("Relationship Coverage",   rel_coverage_score,    rel_cov_suggestions),
]

rows = []
all_suggestions = []

for cat, score, suggestions in categories:
    rows.append({
        "Category": cat,
        "Score /100": f"{score:.0f}",
        "Status": "PASS" if score >= 80 else ("WARN" if score >= 60 else "FAIL"),
    })
    for s in suggestions:
        all_suggestions.append(f"[{cat}] {s}")

overall = round(sum(score for _, score, _ in categories) / NUM_CATEGORIES, 1)
grade = 'A' if overall >= 90 else 'B' if overall >= 80 else 'C' if overall >= 70 else 'D' if overall >= 60 else 'F'

df_report = pd.DataFrame(rows)
print(f"\nOverall AI Readiness Score: {overall}/100 (Grade: {grade})")
print(f"(Simple average of {NUM_CATEGORIES} categories, each scored 0–100)")
display(df_report)

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 46, Finished, Available, Finished, False)


Overall AI Readiness Score: 92.3/100 (Grade: A)
(Simple average of 7 categories, each scored 0–100)


,Category,Score /100,Status
0,Description Coverage,100,PASS
1,Naming Quality,92,PASS
2,Relationship Health,85,PASS
3,DAX Quality,81,PASS
4,Column Metadata,97,PASS
5,Model Structure,90,PASS
6,Relationship Coverage,100,PASS


In [45]:
# ============================================================
# Visual Scorecard
# ============================================================
def score_color(score):
    if score >= 80: return '#2ecc71'   # green
    if score >= 60: return '#f39c12'   # orange
    return '#e74c3c'                   # red

color = score_color(overall)

# Equal-width bars for each category
bar_width = 99 / NUM_CATEGORIES
bar_sections = ""
for cat, score, _ in categories:
    c = score_color(score)
    label = cat[:8]
    bar_sections += (
        f'<div style="width:{bar_width:.1f}%;background:{c};height:30px;display:inline-block;'
        f'text-align:center;color:white;font-size:9px;line-height:30px;overflow:hidden;" '
        f'title="{cat}: {score:.0f}/100">{label}</div>'
    )

html = f"""
<div style="border:2px solid {color};border-radius:12px;padding:20px;margin:10px 0;max-width:900px;">
  <h2 style="color:{color};margin:0;">AI Readiness Score: {overall}/100 - Grade {grade}</h2>
  <p style="color:#666;">Model: {DATASET_NAME} | Workspace: {WORKSPACE_NAME}</p>
  <p style="color:#666;">Assessed: {datetime.now().strftime('%Y-%m-%d %H:%M')} | {NUM_CATEGORIES} categories, each 0–100, simple average</p>
  <div style="background:#eee;border-radius:6px;overflow:hidden;margin:10px 0;">
    {bar_sections}
  </div>
  <table style="width:100%;border-collapse:collapse;margin-top:10px;font-size:12px;">
    <tr style="background:#2F5496;color:white;"><th style="text-align:left;padding:6px;">#</th><th style="text-align:left;padding:6px;">Category</th><th style="text-align:center;">Score /100</th><th style="text-align:center;">Status</th></tr>
    {''.join(f'<tr style="background:{"#f2f2f2" if i%2==0 else "white"};"><td style="padding:5px;">{i+1}</td><td style="padding:5px;">{cat}</td><td style="text-align:center;font-weight:bold;color:{score_color(score)};">{score:.0f}</td><td style="text-align:center;">{"PASS" if score >= 80 else ("WARN" if score >= 60 else "FAIL")}</td></tr>' for i, (cat, score, _) in enumerate(categories))}
    <tr style="background:#2F5496;color:white;font-weight:bold;"><td style="padding:6px;"></td><td style="padding:6px;">OVERALL (Average)</td><td style="text-align:center;">{overall:.1f}</td><td style="text-align:center;">{grade}</td></tr>
  </table>
</div>
"""
display(HTML(html))

# Suggestions
if all_suggestions:
    print("\\nTop Suggestions:")
    for s in all_suggestions[:10]:
        print(f"  • {s}")

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 47, Finished, Available, Finished, False)

#,Category,Score /100,Status
1,Description Coverage,100,PASS
2,Naming Quality,92,PASS
3,Relationship Health,85,PASS
4,DAX Quality,81,PASS
5,Column Metadata,97,PASS
6,Model Structure,90,PASS
7,Relationship Coverage,100,PASS
,OVERALL (Average),92.3,A


\nTop Suggestions:
  • [Naming Quality] 33 objects have technical or unfriendly names.
  • [Relationship Health] 1 BiDi relationship(s) detected via heuristic.
  • [DAX Quality] 11 of 32 measures have no format string.
  • [DAX Quality]   - Non-Commercial Latest Snapshot Date (no format string)
  • [DAX Quality]   - Non-Commercial Snapshot Refresh Gap Days (no format string)
  • [DAX Quality]   - Commercial Current Snapshot Incidents Per Day (no format string)
  • [DAX Quality] 13 DAX expression(s) have issues:
  • [DAX Quality]   - [Measure] Commercial Volume Lock Variance Percentage vs Previous Snapshot: uses / instead of DIVIDE(), FILTER on full table
  • [DAX Quality]   - [Measure] Commercial Volume Lock Difference vs Previous Snapshot: FILTER on full table
  • [DAX Quality]   - [Measure] Commercial Volume Lock Variance Percentage vs Previous to Previous Snapshot: uses / instead of DIVIDE(), FILTER on full table


In [46]:
# ============================================================
# Alert check
# ============================================================
is_alert = overall < ALERT_THRESHOLD

if is_alert:
    alert_msg = (
        f" ALERT: AI Readiness Score for '{DATASET_NAME}' is {overall}/100 (Grade {grade}), "
        f"below the threshold of {ALERT_THRESHOLD}."
    )
    display(HTML(f'<div style="background:#fff3cd;border:1px solid #ffc107;padding:15px;border-radius:8px;">{alert_msg}</div>'))
    print("\nPriority improvements:")
    for s in all_suggestions[:5]:
        print(f"  → {s}")
else:
    display(HTML(f'<div style="background:#d4edda;border:1px solid #28a745;padding:15px;border-radius:8px;"> AI Readiness Score is {overall}/100 — above the alert threshold of {ALERT_THRESHOLD}.</div>'))

StatementMeta(, ce0ea24e-bdc7-401a-98cf-475af89e7d21, 48, Finished, Available, Finished, False)